### **Import libraries**: 

In [1]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import sys

### **Load Parquet file into a Pandas DataFrame object and calculate mean sentiment, sentiment standard deviation, number of articles, number of strongly negative articles, and sentiment momentum.**

In [2]:
def load_and_calculate(parquet_file_name_param):
    df = pd.read_parquet(parquet_file_name_param)
    tickers = list(set(df.index.get_level_values(0)))
    tickers.sort()
    dates = list(set(df.index.get_level_values(1)))
    dates = [str(element) for element in dates]
    dates.sort()
    date_today = str(datetime.today() - timedelta(days=2))[:10]
    iterables = [tickers, pd.to_datetime(dates)]
    index = pd.MultiIndex.from_product(iterables, names=["Ticker", "Timestamp"])
    df_aggregated = pd.DataFrame(index = index, columns = ["sent_mean", "sent_std", "news_count", "pct_strong_negative", "sent_momentum"])
    df_aggregated.sort_index(inplace = True, level = 1, ascending = False, sort_remaining = False)
    
    df_aggregated.loc[("AAPL", "2026-04-04"), df_aggregated.columns] = 5
    #print(df_aggregated)

    #print(df_aggregated)
    #df_aggregated = pd.DataFrame(index = tickers, columns = ["date", "sent_mean", "sent_std", "news_count", "pct_strong_negative", "sent_momentum"])
    
    #df_aggregated = df.copy()
    #df_aggregated.drop(df_aggregated.columns, axis = 1, inplace = True)
    #df_aggregated[["sent_mean", "sent_std", "news_count", "pct_strong_negative", "sent_momentum"]] = np.nan
    #df_aggregated.index = df_aggregated.index.droplevel(level=2)
    #df_aggregated.index.drop_duplicated()
    #print(df_aggregated)
    #print(df.loc["AAPL", date_today]["pos"].sub(df.loc["AAPL", date_today]["neg"], axis = 0))
    #sent_today = df.loc["AAPL", "2026-04-04"]["pos"].sub(df.loc["AAPL", "2026-04-04"]["neg"], axis = 0)
    #sent_mean_today = sent_today.mean()
    #sent_std_today = sent_today.std()
    #news_count_today = len(sent_today)
    #pct_strong_negative_today = len(df.loc["AAPL", "2026-04-04"]["neg"][df.loc["AAPL", "2026-04-04"]["neg"]>0.7]) / len(df.loc["AAPL", "2026-04-04"]["neg"])
    #send_5_day_avg = np.mean([(df.loc["AAPL", pd.Timestamp("2026-04-04") - timedelta(days = i)]["pos"].sub(df.loc["AAPL", pd.Timestamp("2026-04-04") - timedelta(days = i)]["neg"], axis = 0)).mean() for i in range(5)])
    #print(send_5_day_avg)
    for ticker in tickers:
        for date in dates:
            try:
                sent_diff = df.loc[ticker, date]["pos"].sub(df.loc[ticker, date]["neg"], axis = 0)
                sent_mean_today = sent_diff.mean()
                sent_std_today = sent_diff.std()
                news_count_today = len(sent_diff)
                if news_count_today == 1:
                    sent_std_today = 0
                pct_strong_negative_today = (len(df.loc[ticker, date]["neg"][df.loc[ticker, date]["neg"]>0.7]) / len(df.loc[ticker, date]["neg"]))*100
                try:
                    sent_5_day_avg = np.mean([(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["pos"].sub(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["neg"], axis = 0)).mean() for i in range(5)])
                    sent_momentum = sent_mean_today - sent_5_day_avg

                    df_aggregated.loc[(ticker, date), df_aggregated.columns] = sent_mean_today, sent_std_today, news_count_today, pct_strong_negative_today, sent_momentum
                    
                    #print("in")
                    #print("hi")
                    #df_aggregated["sent_mean"].loc[ticker, date] = 5
                    
                    #break
                    #print("Sent: " + str(round(sent_mean_today, 2)), "Sent_std: " + str(round(sent_std_today, 2)), "News_count: " + str(round(news_count_today, 2)), "Very negative %: " + str(round(pct_strong_negative_today, 2)), "Sent_momentum: " + str(round(sent_momentum, 2)))
            
                except KeyError:
                    
                    df_aggregated.loc[(ticker, date), df_aggregated.columns] = sent_mean_today, sent_std_today, news_count_today, pct_strong_negative_today, np.nan
                    #df_aggregated["sent_mean"].loc[ticker, date] = 5
                    #print(df_aggregated)
                    #print("in")
                    #print(df_aggregated.loc[ticker, date])
                    #break
                    #print("Sent: " + str(round(sent_mean_today, 2)), "Sent_std: " + str(round(sent_std_today, 2)), "News_count: " + str(round(news_count_today, 2)), "Very negative %: " + str(round(pct_strong_negative_today, 2)))
                    #print("No article was published for at least one of the five previous days for " + str(ticker) + ".")
                    #print("Sent: " + str(round(sent_mean_today, 2)), "Sent_std: " + str(round(sent_std_today, 2)), "News_count: " + str(round(news_count_today, 2)), "Very negative %: " + str(round(pct_strong_negative_today, 2)))
                
            except KeyError:
                pass
                #print("No article was published about " + str(ticker) + " today.")
    
    print(df_aggregated)     

### **Main program used to collectively execute the whole script.**

In [3]:
def main():
    parquet_file_name = "daily_ticker_headlines_with_FinBERT_scoring.parquet"
    df_with_alt_data_features = load_and_calculate(parquet_file_name)
main()

/var/folders/90/y97m2k6n2z745vb16qxrb4cc0000gn/T/ipykernel_15630/3777375765.py:37: PerformanceWarning: indexing past lexsort depth may impact performance.
  sent_diff = df.loc[ticker, date]["pos"].sub(df.loc[ticker, date]["neg"], axis = 0)
/var/folders/90/y97m2k6n2z745vb16qxrb4cc0000gn/T/ipykernel_15630/3777375765.py:43: PerformanceWarning: indexing past lexsort depth may impact performance.
  pct_strong_negative_today = (len(df.loc[ticker, date]["neg"][df.loc[ticker, date]["neg"]>0.7]) / len(df.loc[ticker, date]["neg"]))*100
/var/folders/90/y97m2k6n2z745vb16qxrb4cc0000gn/T/ipykernel_15630/3777375765.py:45: PerformanceWarning: indexing past lexsort depth may impact performance.
  sent_5_day_avg = np.mean([(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["pos"].sub(df.loc[ticker, pd.Timestamp(date) - timedelta(days = i)]["neg"], axis = 0)).mean() for i in range(5)])


                  sent_mean  sent_std news_count pct_strong_negative  \
Ticker Timestamp                                                       
AAPL   2026-04-04  0.090954  0.233888          5                 0.0   
AMD    2026-04-04  0.449302  0.412799          3                 0.0   
AMZN   2026-04-04  0.084157  0.352984         13                 0.0   
BABA   2026-04-04 -0.018542         0          1                 0.0   
BAC    2026-04-04 -0.256606  0.447565          2                 0.0   
...                     ...       ...        ...                 ...   
TSM    2026-03-31  0.066644  0.532387          7           14.285714   
UNH    2026-03-31       NaN       NaN        NaN                 NaN   
V      2026-03-31  0.094814         0          1                 0.0   
WMT    2026-03-31       NaN       NaN        NaN                 NaN   
XOM    2026-03-31 -0.178109  0.778681          5                40.0   

                  sent_momentum  
Ticker Timestamp             